# Libraries

In [1]:
import keras
import keras.layers as lay
import tensorflow as tf

import numpy as np
import matplotlib.pyplot as plt
import multiprocessing

from src.functions import *

# Defines

In [2]:
TAM_POP = 100

# Dataset load

In [3]:
(trainx,trainy),(valx,valy) = keras.datasets.mnist.load_data()

# Model Instantiation

In [4]:
model = keras.Sequential([
    lay.InputLayer((28,28)),
    lay.Rescaling(1./127.5,-1),
    lay.Flatten(),
    # lay.Dense(2**8,activation='relu'),
    lay.Dense(2**8,activation='relu'),
    lay.Dense(10,activation='softmax')
],name=f'Model_Base')

model.compile(
    optimizer=keras.optimizers.SGD(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

# Generation of N chromosomes

In [5]:
cromo = tf.random.uniform((TAM_POP,*p2v(model.get_weights()).shape),-1,1)
print(cromo.shape)

(100, 203530)


# Multi-process training (change the number of processes)

In [19]:
cromos_list = [(cromo[i],model,trainx,trainy) for i in range(len(cromo))]
if __name__ == '__main__':
    with multiprocessing.Pool(processes=1) as pool:
        accs = pool.map(avaliar,cromos_list)

# Show accuracy and save the weights of the best model

In [20]:
print(np.argsort(accs)[[-1,-2]])
print(accs[np.argsort(accs)[-1]])

indices = np.argsort(accs)[[-1,-2]]
pais = [cromo[indices[0]],cromo[indices[1]]]
model.set_weights(v2p(pais[0],model.get_weights()))
keras.Model.save_weights(model,'best_weights.weights.h5')

[90 13]
0.19798333942890167


# Crossover the parents and add mutations to the offspring.
### *Also, put the two best parents at the top of the list.*

In [18]:
cromo = []
for _ in range(0,TAM_POP,2):
    filho1, filho2 = crossover(pais[0],pais[1])
    cromo.append(mutacao(filho1))
    cromo.append(mutacao(filho2))

cromo[0] = pais[0]
cromo[1] = pais[1]